# In Search of Hallucinations: GPT-2 Attention Analysis
## Master Notebook — Kaggle Edition
Structured companion to the blog series. Runs on Kaggle (GPU recommended).

### Sections
1. Setup & model loading
2. Algorithm verification (Gini, Entropy — known-answer tests)
3. Layer indexing verification
4. Core extraction functions
5. Plausible false pairs — structural metrics (Blog §II)
6. Probability/confidence analysis (Blog §III)
7. Organic failure — math problems (Blog §IV)
8. Baselines — easy questions and simple arithmetic (Blog §IV)
9. Historical questions (Blog §IV)
10. Long-form generation with real-time entropy tracking (Blog §V)
11. GPT-2 Large replication (Blog §VI)
── PLACEHOLDERS (fill in from other notebooks) ──
12. Forced false completions (Blog §III)
13. Continuation cascade testing (Blog §III)
14. Paraphrasing tests (Blog §III)
15. Qwen2.5 testing (Blog §VI)

In [ ]:
# ── LIBRARY INSTALLATION ────────────────────────────────────────────────
# Run this cell first on a fresh Kaggle session.
# Most packages are pre-installed on Kaggle; the two below may not be.
# sentence-transformers is optional: if unavailable, the continuation
# cascade section falls back to lexical metrics only.

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

install('sentence-transformers')   # continuation cascade semantic similarity
install('scikit-learn')            # cosine_similarity used in cascade analysis

print('Installation complete.')

In [ ]:
# ── SECTION 1: SETUP & IMPORTS ─────────────────────────────────────────
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import jensenshannon
from transformers import GPT2LMHeadModel, GPT2Tokenizer, AutoModelForCausalLM, AutoTokenizer
import re
import warnings
warnings.filterwarnings('ignore')

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

In [ ]:
# ── SECTION 1 (cont): MODEL LOADING ────────────────────────────────────
# Primary model for most experiments is GPT-2 Medium (345M parameters).
# GPT-2 Medium has 24 layers (indexed 0-23), 16 attention heads.
# GPT-2 Large (774M) is used in Section 11 for replication testing.
# eager attention implementation is required to extract attention weights;
# the default SDPA implementation does not return them.

def load_gpt2(model_name='gpt2-medium'):
    print(f'Loading {model_name}...')
    model = GPT2LMHeadModel.from_pretrained(
        model_name,
        attn_implementation='eager'
    ).to(DEVICE)
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.config.output_attentions = True
    model.eval()
    print(f'  Layers : {model.config.n_layer}')
    print(f'  Heads  : {model.config.n_head}')
    print(f'  Attn   : {model.config._attn_implementation}')
    return model, tokenizer

model, tokenizer = load_gpt2('gpt2-medium')
N_LAYERS = model.config.n_layer   # 24 for gpt2-medium
N_HEADS  = model.config.n_head    # 16 for gpt2-medium
print(f'\nN_LAYERS={N_LAYERS}, N_HEADS={N_HEADS}')

In [ ]:
# ── SECTION 2: ALGORITHM VERIFICATION ──────────────────────────────────
# Before running any experiments we verify that Gini and Entropy
# implementations return correct values on inputs where we know the answer.
#
# Gini coefficient:
#   - Uniform distribution  → 0.0  (perfectly equal)
#   - Concentrated (all weight on one token) → (n-1)/n ≈ 1.0 for large n
#
# Shannon entropy (nats):
#   - Uniform over n tokens → ln(n)  (maximum entropy)
#   - Delta distribution    → 0.0   (minimum entropy)
#
# If either test fails the implementations must be corrected before
# any experimental results can be trusted.

def gini(x: torch.Tensor) -> float:
    """Gini coefficient of a 1-D probability distribution.

    Formula: G = (n+1 - 2*sum(cumsum)) / n
    where cumsum is the cumulative sum of x sorted ascending.
    Works correctly when x sums to 1 (attention weights always do).
    Returns a value in [0, (n-1)/n].
    """
    sorted_x = torch.sort(x)[0]
    n = len(sorted_x)
    cumsum = torch.cumsum(sorted_x, dim=0)
    return ((n + 1 - 2 * torch.sum(cumsum) / cumsum[-1]) / n).item()


def entropy(x: torch.Tensor) -> float:
    """Shannon entropy (nats) of a 1-D probability distribution.

    Formula: H = -sum(p * log(p))
    Epsilon added for numerical stability; input is renormalised first.
    Returns 0.0 for a delta distribution and ln(n) for uniform.
    """
    x = x + 1e-10
    x = x / x.sum()
    return (-(x * torch.log(x)).sum()).item()


# ── Verification tests ──────────────────────────────────────────────────
def verify_algorithms():
    passed = True

    for n in [4, 10, 100]:
        # Gini: uniform → 0
        uniform = torch.ones(n) / n
        g = gini(uniform)
        expected_g = 0.0
        ok = abs(g - expected_g) < 1e-5
        print(f'Gini uniform  n={n:3d}: {g:.6f}  expected {expected_g:.6f}  {"PASS" if ok else "FAIL"}')
        passed = passed and ok

        # Gini: concentrated → (n-1)/n
        delta = torch.zeros(n); delta[-1] = 1.0
        g = gini(delta)
        expected_g = (n - 1) / n
        ok = abs(g - expected_g) < 1e-5
        print(f'Gini delta    n={n:3d}: {g:.6f}  expected {expected_g:.6f}  {"PASS" if ok else "FAIL"}')
        passed = passed and ok

        # Entropy: uniform → ln(n)
        h = entropy(uniform)
        expected_h = np.log(n)
        ok = abs(h - expected_h) < 1e-4
        print(f'Entropy uniform n={n:3d}: {h:.6f}  expected {expected_h:.6f}  {"PASS" if ok else "FAIL"}')
        passed = passed and ok

        # Entropy: delta → 0
        h = entropy(delta)
        ok = abs(h - 0.0) < 1e-4
        print(f'Entropy delta   n={n:3d}: {h:.6f}  expected 0.000000  {"PASS" if ok else "FAIL"}')
        passed = passed and ok
        print()

    print('=' * 50)
    print('ALL ALGORITHM TESTS PASSED' if passed else 'WARNING: ALGORITHM TESTS FAILED — do not trust results')
    return passed

algo_ok = verify_algorithms()

In [ ]:
# ── SECTION 3: LAYER INDEXING VERIFICATION ──────────────────────────────
# GPT-2 uses 0-based layer indexing.
# outputs.attentions is a tuple of length N_LAYERS.
# outputs.attentions[k] is the attention tensor for layer k (0-indexed).
# This cell confirms that when we request layer k we actually get layer k,
# and prints the tensor shapes so we can reason about dimensions.
#
# Expected shapes for gpt2-medium, sequence length L:
#   attentions[k] : (batch=1, heads=16, L, L)
#   hidden_states[k]: (batch=1, L, hidden=1024)
#   (hidden_states has N_LAYERS+1 entries: embedding + one per layer)

def verify_layer_indexing(text='The cat sat on the mat'):
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    L = len(tokens)

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True, output_hidden_states=True)

    print(f'Text  : "{text}"')
    print(f'Tokens: {tokens}')
    print(f'Sequence length L = {L}')
    print()
    print(f'Number of attention tensors returned : {len(outputs.attentions)}')
    print(f'Expected (= N_LAYERS)                : {N_LAYERS}')
    print(f'Match: {len(outputs.attentions) == N_LAYERS}')
    print()
    for k in [0, 11, 22, N_LAYERS - 1]:
        shape = tuple(outputs.attentions[k].shape)
        print(f'  outputs.attentions[{k:2d}].shape = {shape}  '
              f'(batch, heads={shape[1]}, L={shape[2]}, L={shape[3]})')
    print()
    print(f'Number of hidden_state tensors       : {len(outputs.hidden_states)}')
    print(f'Expected (= N_LAYERS + 1)            : {N_LAYERS + 1}')
    hs_shape = tuple(outputs.hidden_states[0].shape)
    print(f'  hidden_states[0].shape = {hs_shape}  (embedding layer)')
    print()
    print('Layer mapping summary:')
    print(f'  outputs.attentions[0]          → transformer block 0 (first)')
    print(f'  outputs.attentions[{N_LAYERS-1}]         → transformer block {N_LAYERS-1} (last)')
    print(f'  Experiments use layers 18-23   → valid range (0 to {N_LAYERS-1})')

verify_layer_indexing()

In [ ]:
# ── SECTION 4: CORE EXTRACTION FUNCTIONS ───────────────────────────────
# These are used by all subsequent experiments.

def get_all_outputs(text, mdl, tok):
    """Return attentions, hidden states, logits, and tokens for a text.

    Shapes (gpt2-medium, sequence length L):
      attentions   : (N_LAYERS, heads, L, L)  — batch dim removed
      hidden_states: (N_LAYERS+1, L, hidden)  — batch dim removed
      logits       : (L, vocab)
    """
    inputs = tok(text, return_tensors='pt').to(DEVICE)
    tokens = tok.convert_ids_to_tokens(inputs['input_ids'][0])
    with torch.no_grad():
        out = mdl(**inputs, output_attentions=True, output_hidden_states=True)
    return {
        'tokens'       : tokens,
        'attentions'   : torch.stack(out.attentions)[:, 0],      # (layers, heads, L, L)
        'hidden_states': torch.stack(out.hidden_states)[:, 0],   # (layers+1, L, hidden)
        'logits'       : out.logits[0],                          # (L, vocab)
    }


def compute_layer_entropy(att_layer):
    """Mean and per-head Shannon entropy for one layer's attention tensor.

    att_layer shape: (heads, L, L)
    Each row of the attention matrix is a distribution over L tokens.
    Returns:
      mean_entropy        : scalar averaged over heads and positions
      head_entropy_var    : variance across heads (head disagreement proxy)
      per_head_mean       : (heads,) array — individual head mean entropies
    """
    att = att_layer + 1e-10
    att = att / att.sum(dim=-1, keepdim=True)              # (heads, L, L)
    h   = -(att * torch.log(att)).sum(dim=-1)              # (heads, L)
    per_head_mean    = h.mean(dim=-1)                      # (heads,)
    mean_entropy     = per_head_mean.mean().item()
    head_entropy_var = per_head_mean.var().item()
    return mean_entropy, head_entropy_var, per_head_mean.cpu().numpy()


def compute_layer_gini(att_layer):
    """Mean Gini concentration for one layer's attention tensor.

    att_layer shape: (heads, L, L)
    Computes Gini coefficient for each (head, position) attention row,
    then returns the mean.
    """
    concentrations = []
    for h_idx in range(att_layer.shape[0]):
        for pos in range(att_layer.shape[1]):
            concentrations.append(gini(att_layer[h_idx, pos]))
    return float(np.mean(concentrations))


def compute_hidden_variance(hs_layer):
    """Mean activation variance across hidden dimensions for one layer.

    hs_layer shape: (L, hidden)
    Variance taken across the hidden dimension for each position,
    then averaged across positions.
    """
    return hs_layer.var(dim=-1).mean().item()

In [ ]:
# ── SECTION 5: PLAUSIBLE FALSE PAIRS (Blog §II) ─────────────────────────
# Three categories designed to test different detection scenarios.
# Each tuple is (true_statement, false_statement).
# Category 1: Subtle errors — same domain, plausible confusion.
# Category 2: Nonsense control — semantically incoherent pairs.
# Category 3: Contested facts — genuinely debated claims.
# Testing each category in both orderings controls for position bias.

PLAUSIBLE_FALSE_PAIRS = [
    ('Bell invented the telephone',              'Bell invented the telegraph'),
    ('Einstein won Nobel for photoelectric effect', 'Einstein won Nobel for relativity'),
    ('Fleming discovered penicillin in 1928',    'Fleming discovered penicillin in 1927'),
    ('Darwin wrote Origin of Species',           'Darwin wrote Descent of Man'),
    ('Columbus sailed in 1492',                  'Columbus sailed in 1493'),
    ('Watson and Crick discovered DNA structure','Watson alone discovered DNA structure'),
    ('Edison invented the light bulb',           'Edison invented the phonograph'),
    ('Curie discovered radium',                  'Curie discovered polonium'),
    ('Newton published Principia in 1687',       'Newton published Principia in 1686'),
    ('Wright brothers flew in 1903',             'Wright brothers flew in 1902'),
    ('Titanic sank in 1912',                     'Titanic sank in 1911'),
    ('WWII ended in 1945',                       'WWII ended in 1944'),
    ('Moon landing was in 1969',                 'Moon landing was in 1968'),
    ('Berlin Wall fell in 1989',                 'Berlin Wall fell in 1988'),
    ('Washington was first US president',        'Jefferson was first US president'),
    ('Beethoven wrote 9 symphonies',             'Beethoven wrote 10 symphonies'),
    ('Shakespeare died in 1616',                 'Shakespeare died in 1615'),
    ('Lincoln gave Gettysburg Address in 1863',  'Lincoln gave Gettysburg Address in 1864'),
    ('Marie Curie discovered radium',            'Marie Curie discovered DNA'),
    ('The Nile is the longest river',            'The Amazon is the longest river'),
]

NONSENSE_PAIRS = [
    ('Green ideas sleep furiously',              'Colorless ideas sleep furiously'),
    ('The table laughed at midnight',            'The chair laughed at midnight'),
    ('Tuesday tastes like purple sounds',        'Wednesday tastes like purple sounds'),
    ('Gravity speaks in ancient colors',         'Gravity speaks in modern colors'),
    ('Time eats the frozen shadows',             'Time eats the melted shadows'),
    ('Numbers dance with invisible music',       'Numbers dance with visible music'),
    ('Silence weighs more than thunder',         'Silence weighs less than thunder'),
    ('Rain calculates the meaning of blue',      'Rain calculates the meaning of red'),
    ('The sun argues with square circles',       'The sun argues with round circles'),
    ('Memory flows backwards through glass',     'Memory flows forwards through glass'),
]

CONTESTED_PAIRS = [
    ('Columbus discovered America',              'Leif Erikson discovered America'),
    ('Pluto is a planet',                        'Pluto is a dwarf planet'),
    ('Tomatoes are vegetables',                  'Tomatoes are fruits'),
    ('Glass is a liquid',                        'Glass is a solid'),
    ('Dinosaurs were cold-blooded',              'Dinosaurs were warm-blooded'),
    ('Mount Everest is tallest from sea level',  'Mauna Kea is tallest from base'),
    ('The Great Wall is visible from space',     'The Great Wall is not visible from space'),
    ('Lightning never strikes twice',            'Lightning can strike the same place twice'),
    ('Goldfish have three-second memory',        'Goldfish have longer memory'),
    ('Humans only use 10 percent of brain',      'Humans use all of their brain'),
]

In [ ]:
# ── SECTION 5 (cont): ANALYSIS FUNCTION ────────────────────────────────

ANALYSIS_LAYERS = list(range(18, 24))  # Layers 18-23 (upper-middle of 24 total)

def analyze_pair(true_text, false_text, mdl, tok, layers=ANALYSIS_LAYERS):
    """Compute structural metrics for one true/false pair across specified layers.

    Returns a list of dicts, one per (statement, layer) combination.
    """
    rows = []
    for label, text in [('true', true_text), ('false', false_text)]:
        out = get_all_outputs(text, mdl, tok)
        for layer in layers:
            att = out['attentions'][layer]         # (heads, L, L)
            hs  = out['hidden_states'][layer + 1]  # +1 because index 0 is embedding
            mean_ent, head_var, per_head = compute_layer_entropy(att)
            rows.append({
                'label'          : label,
                'text'           : text,
                'layer'          : layer,
                'entropy'        : mean_ent,
                'head_entropy_var': head_var,
                'gini'           : compute_layer_gini(att),
                'hidden_variance': compute_hidden_variance(hs),
            })
    return rows


def run_pairs_experiment(pairs, mdl, tok, label='experiment'):
    """Run analyze_pair over a list of (true, false) tuples.

    Returns a DataFrame and prints a statistical summary.
    """
    all_rows = []
    for i, (true_text, false_text) in enumerate(pairs):
        print(f'  Pair {i+1}/{len(pairs)}: {true_text[:45]}...')
        all_rows.extend(analyze_pair(true_text, false_text, mdl, tok))

    df = pd.DataFrame(all_rows)

    print(f'\n{"="*60}')
    print(f'RESULTS: {label}')
    print(f'{"="*60}')
    for metric in ['entropy', 'gini', 'hidden_variance', 'head_entropy_var']:
        true_vals  = df[df['label'] == 'true'][metric]
        false_vals = df[df['label'] == 'false'][metric]
        t, p = stats.ttest_rel(true_vals.values, false_vals.values)
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f'{metric:20s}  true={true_vals.mean():.4f}  false={false_vals.mean():.4f}  '
              f't={t:6.3f}  p={p:.4f} {sig}')
    return df

In [ ]:
# Run plausible pairs — three categories, both orderings
print('Running PLAUSIBLE FALSE PAIRS...')
df_plausible = run_pairs_experiment(PLAUSIBLE_FALSE_PAIRS, model, tokenizer,
                                    label='Plausible False Pairs')
df_plausible.to_csv('results_plausible_pairs.csv', index=False)

print('\nRunning PLAUSIBLE PAIRS FLIPPED (position bias check)...')
flipped = [(f, t) for t, f in PLAUSIBLE_FALSE_PAIRS]
df_plausible_flip = run_pairs_experiment(flipped, model, tokenizer,
                                         label='Plausible Pairs (flipped)')

print('\nRunning NONSENSE CONTROL PAIRS...')
df_nonsense = run_pairs_experiment(NONSENSE_PAIRS, model, tokenizer,
                                   label='Nonsense Control')
df_nonsense.to_csv('results_nonsense_pairs.csv', index=False)

print('\nRunning CONTESTED PAIRS...')
df_contested = run_pairs_experiment(CONTESTED_PAIRS, model, tokenizer,
                                    label='Contested Facts')
df_contested.to_csv('results_contested_pairs.csv', index=False)

In [ ]:
# ── SECTION 6: PROBABILITY / CONFIDENCE ANALYSIS (Blog §III) ───────────
# Tests whether output probability (softmax of final-token logits)
# reliably distinguishes true from false statements.
# Also examines distribution shape: top-1, top-5, output entropy.

def output_confidence(logits_last):
    """Metrics derived from the final token's output distribution.

    logits_last: 1-D tensor of shape (vocab,)
    """
    probs = F.softmax(logits_last, dim=0)
    sorted_p, _ = torch.sort(probs, descending=True)
    return {
        'top1_prob'       : sorted_p[0].item(),
        'top5_prob_sum'   : sorted_p[:5].sum().item(),
        'output_entropy'  : entropy(probs),
        'top1_to_top2'    : (sorted_p[0] / (sorted_p[1] + 1e-10)).item(),
    }


def run_confidence_experiment(pairs, mdl, tok):
    """For each pair check whether higher model probability matches true statement."""
    rows = []
    for true_text, false_text in pairs:
        out_t = get_all_outputs(true_text,  mdl, tok)
        out_f = get_all_outputs(false_text, mdl, tok)
        conf_t = output_confidence(out_t['logits'][-1])
        conf_f = output_confidence(out_f['logits'][-1])
        rows.append({
            'true_text'       : true_text,
            'false_text'      : false_text,
            'true_top1'       : conf_t['top1_prob'],
            'false_top1'      : conf_f['top1_prob'],
            'true_higher_prob': conf_t['top1_prob'] > conf_f['top1_prob'],
            'true_entropy'    : conf_t['output_entropy'],
            'false_entropy'   : conf_f['output_entropy'],
        })
    df = pd.DataFrame(rows)
    pct = df['true_higher_prob'].mean() * 100
    print(f'True statement had higher output probability: {pct:.1f}% of pairs')
    print(f'(Random chance = 50%. If near 50% confidence is not a reliable signal.)')
    return df

print('Running confidence analysis on plausible pairs...')
df_confidence = run_confidence_experiment(PLAUSIBLE_FALSE_PAIRS, model, tokenizer)
df_confidence.to_csv('results_confidence.csv', index=False)

In [ ]:
# ── SECTION 7: ORGANIC FAILURE — MATH PROBLEMS (Blog §IV) ──────────────
# GPT-2 cannot perform 6×6 digit multiplication. These problems are chosen
# to guarantee failure, providing a ground-truth hallucination scenario.
# We track entropy and other signals during the attempted computation.

MATH_PROBLEMS = [
    (624710, 945711, 590795118810),
    (382267, 369301, 141171585367),
    (780802, 715724, 558838730648),
    (899384, 479354, 431123317936),
    (427830, 979273, 418962367590),
    (468673, 475095, 222664198935),
    (857113, 310554, 266179870602),
    (407154, 612204, 249261307416),
    (342957, 558068, 191393327076),
    (775247, 459263, 356042262961),
    (601014, 766083, 460426608162),
    (591401, 497236, 294065867636),
    (748616, 834241, 624526160456),
    (881027, 517051, 455535891377),
    (692366, 929258, 643386644428),
    (840358, 377170, 316957826860),
    (622392, 557946, 347261126832),
    (767519, 718932, 551793969708),
    (681380, 960751, 654636516380),
]

def run_math_experiment(problems, mdl, tok, layers=ANALYSIS_LAYERS):
    rows = []
    for i, (n1, n2, correct) in enumerate(problems):
        prompt = f': {n1} * {n2} ='
        inputs = tok(prompt, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            output = mdl.generate(
                inputs['input_ids'],
                max_new_tokens=15,
                do_sample=False,
                pad_token_id=tok.pad_token_id
            )
        response = tok.decode(output[0], skip_special_tokens=True)
        answer_str = response[len(prompt):].strip()
        numbers = re.findall(r'\d+', answer_str)
        model_answer = int(numbers[0]) if numbers else None
        is_correct = (model_answer == correct)
        print(f'  [{i+1:2d}] {n1}*{n2}: model={model_answer}  correct={correct}  {"✓" if is_correct else "✗"}')

        out = get_all_outputs(response, mdl, tok)
        for layer in layers:
            att = out['attentions'][layer]
            hs  = out['hidden_states'][layer + 1]
            mean_ent, head_var, _ = compute_layer_entropy(att)
            rows.append({
                'problem_id'     : i,
                'n1': n1, 'n2': n2,
                'correct_answer' : correct,
                'model_answer'   : model_answer,
                'is_correct'     : is_correct,
                'layer'          : layer,
                'entropy'        : mean_ent,
                'head_entropy_var': head_var,
                'gini'           : compute_layer_gini(att),
                'hidden_variance': compute_hidden_variance(hs),
            })
    df = pd.DataFrame(rows)
    accuracy = df.drop_duplicates('problem_id')['is_correct'].mean()
    print(f'\nAccuracy: {accuracy*100:.0f}%')
    print(f'Mean entropy (all layers): {df["entropy"].mean():.4f}')
    return df

print('Running math problems...')
df_math = run_math_experiment(MATH_PROBLEMS, model, tokenizer)
df_math.to_csv('results_math.csv', index=False)

In [ ]:
# ── SECTION 8 & 9: BASELINES AND HISTORICAL QUESTIONS (Blog §IV) ────────

EASY_QUESTIONS = [
    ('Please answer: What color is the sky?',      'blue'),
    ('Please answer: Are dogs animals?',           'yes'),
    ('Please answer: Are peppers spicy?',          'yes'),
    ('Please answer: What is 2 + 2?',              '4'),
    ('Please answer: What do cows produce?',       'milk'),
]

SIMPLE_MATH = [
    ('Please answer: 2 * 3 =',  '6'),
    ('Please answer: 5 * 4 =',  '20'),
    ('Please answer: 10 * 2 =', '20'),
    ('Please answer: 3 * 7 =',  '21'),
    ('Please answer: 6 * 5 =',  '30'),
]

HISTORICAL_QUESTIONS = [
    ('Please answer: Who invented the telephone?',              'Bell'),
    ('Please answer: Who invented the telegraph?',             'Morse'),
    ('Please answer: What did Einstein win the Nobel Prize for?', 'photoelectric'),
    ('Please answer: What book did Darwin write?',             'Origin'),
    ('Please answer: Who was the first US president?',         'Washington'),
]

def run_qa_experiment(questions, mdl, tok, label, layers=ANALYSIS_LAYERS):
    """Generate short answers and collect structural metrics."""
    rows = []
    for i, (prompt, expected) in enumerate(questions):
        inputs = tok(prompt, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            output = mdl.generate(
                inputs['input_ids'],
                max_new_tokens=10,
                do_sample=False,
                pad_token_id=tok.pad_token_id
            )
        response   = tok.decode(output[0], skip_special_tokens=True)
        answer_str = response[len(prompt):].strip()
        is_correct = expected.lower() in answer_str.lower()
        print(f'  [{i+1}] Expected: {expected:15s}  Got: {answer_str[:30]:30s}  {"✓" if is_correct else "✗"}')

        out = get_all_outputs(response, mdl, tok)
        for layer in layers:
            att = out['attentions'][layer]
            hs  = out['hidden_states'][layer + 1]
            mean_ent, head_var, _ = compute_layer_entropy(att)
            rows.append({
                'question_id'    : i,
                'prompt'         : prompt,
                'expected'       : expected,
                'model_answer'   : answer_str,
                'is_correct'     : is_correct,
                'layer'          : layer,
                'entropy'        : mean_ent,
                'head_entropy_var': head_var,
                'gini'           : compute_layer_gini(att),
                'hidden_variance': compute_hidden_variance(hs),
            })
    df = pd.DataFrame(rows)
    accuracy = df.drop_duplicates('question_id')['is_correct'].mean()
    print(f'  Accuracy: {accuracy*100:.0f}%  Mean entropy: {df["entropy"].mean():.4f}')
    return df

print('Easy questions:')
df_easy = run_qa_experiment(EASY_QUESTIONS, model, tokenizer, 'Easy')
df_easy.to_csv('results_easy.csv', index=False)

print('\nSimple arithmetic:')
df_simple_math = run_qa_experiment(SIMPLE_MATH, model, tokenizer, 'SimpleMath')
df_simple_math.to_csv('results_simple_math.csv', index=False)

print('\nHistorical questions:')
df_historical = run_qa_experiment(HISTORICAL_QUESTIONS, model, tokenizer, 'Historical')
df_historical.to_csv('results_historical.csv', index=False)

In [ ]:
# ── SECTION 8/9 (cont): ENTROPY COMPARISON ACROSS CONDITIONS ────────────
# Recreates the gradient described in the blog:
#   easy questions → simple math → hard math
# Expected: entropy rises with task difficulty.

def entropy_summary(label, df):
    # Use layer 22 specifically (as referenced in blog text)
    d = df[df['layer'] == 22]
    print(f'{label:25s}  entropy={d["entropy"].mean():.4f} ± {d["entropy"].std():.4f}')

print('Entropy by condition (layer 22):')
entropy_summary('Easy questions',    df_easy)
entropy_summary('Simple arithmetic', df_simple_math)
entropy_summary('Historical Qs',     df_historical)
entropy_summary('Hard math (6x6)',   df_math)

In [ ]:
# ── SECTION 10: LONG-FORM GENERATION — REAL-TIME ENTROPY (Blog §V) ──────
# Generates 50-100 token continuations and tracks entropy token-by-token.
# High entropy (>1.0) tokens are flagged as potentially unreliable.
# This was the most promising result on GPT-2 Medium.

LONG_FORM_PROMPTS = [
    ('Tell me about the life and achievements of Alexander Graham Bell:', 'Bell biography'),
    ('Explain the history and development of the telegraph:', 'telegraph history'),
    ('Describe Einstein contributions to physics:', 'Einstein science'),
    ('What were the major events in Darwins scientific career?', 'Darwin biography'),
    ('How did the telephone change society in the 19th century?', 'telephone society'),
]

def run_long_form_experiment(prompts, mdl, tok,
                             max_new_tokens=80, track_layer=22,
                             entropy_threshold=1.0):
    """Generate text and record per-token entropy at track_layer.

    Returns a DataFrame with one row per (prompt, generated_position).
    Note: track_layer=22 matches the value used in Blog §V.
    """
    rows = []
    for i, (prompt, topic) in enumerate(prompts):
        print(f'\n[{i+1}] {topic}: {prompt[:55]}...')
        inputs = tok(prompt, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            output = mdl.generate(
                inputs['input_ids'],
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tok.pad_token_id
            )
        full_response = tok.decode(output[0], skip_special_tokens=True)
        print(f'  Generated: {full_response[len(prompt):len(prompt)+80]}...')

        out = get_all_outputs(full_response, mdl, tok)
        prompt_len = len(inputs['input_ids'][0])
        att = out['attentions'][track_layer]  # (heads, L, L)

        for pos in range(prompt_len, len(out['tokens'])):
            # Entropy at this position: distribution over previous tokens
            pos_att = att[:, pos, :]             # (heads, L)
            mean_ent, _, _ = compute_layer_entropy(pos_att.unsqueeze(1))
            rows.append({
                'prompt_id'       : i,
                'topic'           : topic,
                'gen_position'    : pos - prompt_len,
                'token'           : out['tokens'][pos],
                'entropy'         : mean_ent,
                'high_entropy'    : mean_ent > entropy_threshold,
                'prompt_attention': att[:, pos, :prompt_len].mean().item(),
            })

    df = pd.DataFrame(rows)

    print(f'\nHigh-entropy tokens (>{entropy_threshold}): '
          f'{df["high_entropy"].sum()} of {len(df)} '
          f'({df["high_entropy"].mean()*100:.1f}%)')
    return df

print('Running long-form generation experiment...')
df_longform = run_long_form_experiment(LONG_FORM_PROMPTS, model, tokenizer)
df_longform.to_csv('results_longform.csv', index=False)

In [ ]:
# ── SECTION 11: GPT-2 LARGE REPLICATION (Blog §VI) ─────────────────────
# GPT-2 Large has 36 layers (indexed 0-35) and 20 heads.
# Rerun key experiments to test whether Medium's patterns generalise.
# Layers 18-23 on Large correspond to mid-network, not near-final.
# For comparison with Medium we use layers 30-35 (upper sixth of 36).

LARGE_ANALYSIS_LAYERS = list(range(30, 36))

print('Loading GPT-2 Large...')
model_large, tokenizer_large = load_gpt2('gpt2-large')
print(f'Large: {model_large.config.n_layer} layers, {model_large.config.n_head} heads')

# Rerun math problems on Large
print('\nMath problems on GPT-2 Large:')
df_math_large = run_math_experiment(MATH_PROBLEMS, model_large, tokenizer_large)
df_math_large['model'] = 'gpt2-large'
df_math_large.to_csv('results_math_large.csv', index=False)

# Rerun easy questions on Large
print('\nEasy questions on GPT-2 Large:')
df_easy_large = run_qa_experiment(EASY_QUESTIONS, model_large, tokenizer_large,
                                  'Easy (Large)', layers=LARGE_ANALYSIS_LAYERS)

# Compare entropy distributions
print('\nEntropy comparison Medium vs Large (hard math):')
print(f'  Medium: {df_math["entropy"].mean():.4f}')
print(f'  Large : {df_math_large["entropy"].mean():.4f}')

In [ ]:
# ── SECTION 12: FORCED COMPLETIONS — CONCEPTUAL NOTE (Blog §III) ────────
#
# The blog groups several experiments under the heading
# 'Do Artificial Scenarios Produce Genuine Hallucinations?'
# and lists 'Forced completions' as the first bullet.
#
# This is NOT a separate experiment with distinct code.
# It is the conceptual label for the class of tests in Section 5
# (plausible pairs) where complete false sentences were fed to the model
# as fully formed token sequences.  Feeding 'Einstein discovered penicillin'
# all at once is functionally equivalent to forcing the model to process
# a wrong completion — the model has no opportunity to choose otherwise.
#
# BLOG NOTE: The blog should clarify this equivalence to avoid implying
# that a separate forced-completion experiment was run.
# Suggested addition to §III:
#   'Feeding a complete false sentence is equivalent to forcing the wrong
#    completion — the model processes the tokens as given.  This is what
#    the plausible pairs experiment in §II did.'
#
# No additional code required — see Section 5 results.

print('Section 12: see Section 5 (plausible pairs) — same experiment.')

In [ ]:
# ── SECTION 13: CONTINUATION CASCADE TESTING (Blog §III) ───────────────
#
# HYPOTHESIS: If a false premise appears early in generation, subsequent
# tokens might show cumulative degradation — increasing entropy, erratic
# attention — as the model tries to maintain coherence with a false claim.
#
# RESULT (blog): No cascading effects detected.  The model treated false
# premises as given context and generated coherently from them.
#
# BLOG NOTE: With num_samples=3 and temperature=0.7 the sample variance
# is high relative to any true/false signal.  The 'no effect' conclusion
# is directionally correct but statistically weak.  Acknowledge this
# limitation explicitly rather than stating it as a firm negative result.
#
# DEPENDENCY NOTE: sentence-transformers is used for semantic consistency.
# If unavailable the code falls back to lexical metrics only.
#
# FUNCTION NAME NOTE: Original code called get_full_model_outputs() and
# compute_attention_concentration().  Replaced here with this notebook's
# get_all_outputs() and compute_layer_gini() for consistency.

try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
    USE_SEMANTIC = True
    print('sentence-transformers available: semantic similarity enabled.')
except ImportError:
    USE_SEMANTIC = False
    print('sentence-transformers not available: lexical metrics only.')


def _continuation_signals(full_text, premise_text, mdl, tok, layer=21):
    """Attention from continuation back to premise, and Gini concentration.

    Uses get_all_outputs() and compute_layer_gini() from Section 4.
    """
    out = get_all_outputs(full_text, mdl, tok)
    att = out['attentions'][layer]                       # (heads, L, L)
    premise_len = len(tok(premise_text)['input_ids'])
    total_len   = len(out['tokens'])
    if total_len <= premise_len:
        return 0.0, 0.0
    premise_att = att[:, premise_len:, :premise_len].mean().item()
    cont_gini   = compute_layer_gini(att[:, premise_len:, :])
    return premise_att, cont_gini


def _lexical_diversity(texts):
    words = [w for t in texts for w in t.lower().split()]
    return len(set(words)) / len(words) if words else 0.0


def run_continuation_cascade(pairs, mdl, tok,
                              continuation_length=50, num_samples=3):
    """Generate continuations from true/false premises and compare patterns.

    pairs: list of (true_statement, false_statement) tuples.
    Returns a DataFrame with one row per pair.
    """
    if USE_SEMANTIC:
        sem_model = SentenceTransformer('all-MiniLM-L6-v2')

    rows = []
    for i, (true_stmt, false_stmt) in enumerate(pairs):
        print(f'  Pair {i+1}/{len(pairs)}: {true_stmt[:40]}...')
        true_conts, false_conts = [], []

        for s in range(num_samples):
            for stmt, bucket in [(true_stmt, true_conts),
                                  (false_stmt, false_conts)]:
                inp = tok(stmt, return_tensors='pt').to(DEVICE)
                with torch.no_grad():
                    out = mdl.generate(
                        inp['input_ids'],
                        attention_mask=inp['attention_mask'],
                        max_new_tokens=continuation_length,
                        temperature=0.7, do_sample=True,
                        pad_token_id=tok.eos_token_id
                    )
                full = tok.decode(out[0], skip_special_tokens=True)
                bucket.append(full[len(stmt):].strip())

        row = {
            'pair_id'              : i,
            'true_statement'       : true_stmt,
            'false_statement'      : false_stmt,
            'true_lex_diversity'   : _lexical_diversity(true_conts),
            'false_lex_diversity'  : _lexical_diversity(false_conts),
            'true_avg_length'      : np.mean([len(c.split()) for c in true_conts]),
            'false_avg_length'     : np.mean([len(c.split()) for c in false_conts]),
        }

        # Semantic consistency (optional)
        if USE_SEMANTIC and num_samples > 1:
            def _avg_sim(texts):
                emb = sem_model.encode(texts)
                sim = sk_cosine(emb)
                tri = sim[np.triu_indices(len(texts), k=1)]
                return float(tri.mean()) if len(tri) else 0.0
            row['true_semantic_consistency']  = _avg_sim(true_conts)
            row['false_semantic_consistency'] = _avg_sim(false_conts)

        # Attention signals on first continuation
        try:
            t_pa, t_gc = _continuation_signals(
                true_stmt + ' ' + true_conts[0],  true_stmt,  mdl, tok)
            f_pa, f_gc = _continuation_signals(
                false_stmt + ' ' + false_conts[0], false_stmt, mdl, tok)
            row.update({
                'true_premise_attention'  : t_pa,
                'false_premise_attention' : f_pa,
                'premise_attention_diff'  : abs(t_pa - f_pa),
                'true_cont_gini'          : t_gc,
                'false_cont_gini'         : f_gc,
                'cont_gini_diff'          : abs(t_gc - f_gc),
            })
        except Exception as e:
            print(f'    Attention analysis failed: {e}')

        rows.append(row)

    df = pd.DataFrame(rows)

    print(f'\n{"="*60}')
    print('CONTINUATION CASCADE SUMMARY')
    print(f'{"="*60}')
    for col in ['premise_attention_diff', 'cont_gini_diff',
                'true_semantic_consistency', 'false_semantic_consistency']:
        if col in df.columns:
            print(f'  {col}: mean={df[col].mean():.4f}  std={df[col].std():.4f}')
    return df


# Use the plausible pairs subset as test data
CASCADE_PAIRS = PLAUSIBLE_FALSE_PAIRS[:5]   # small subset: cascade is slow
print('Running continuation cascade...')
df_cascade = run_continuation_cascade(CASCADE_PAIRS, model, tokenizer)
df_cascade.to_csv('results_cascade.csv', index=False)

In [ ]:
# ── SECTION 14: PARAPHRASING ROBUSTNESS TESTS (Blog §III) ──────────────
#
# HYPOTHESIS: If structural signals (Gini, hidden variance) detect factual
# content rather than surface phrasing, they should be consistent across
# paraphrases of the same statement.
#
# RESULT (blog): Inconclusive.  Signals varied with phrasing, suggesting
# sensitivity to sentence structure, not just semantic content.
#
# BLOG NOTE: Only 5 pairs tested.  This sample size is too small for any
# firm conclusion.  The result is worth reporting honestly as 'we cannot
# rule out that structural signals are phrasing artefacts' rather than
# 'signals are phrasing artefacts.'
#
# BLOG NOTE: Math equivalency pairs (e.g. '10+3=13' vs '3+10=13') were
# also defined in the original code but are not mentioned in the blog.
# Either include them with results or remove them from the code to avoid
# implying they were analysed.
#
# FUNCTION NAME NOTE: Original code called get_full_model_outputs(),
# compute_attention_concentration(), and compute_hidden_state_variance().
# Replaced here with get_all_outputs(), compute_layer_gini(), and
# compute_hidden_variance() for consistency with this notebook.

PARAPHRASE_BASE = [
    ('Bell invented the telephone',         'Bell invented the telegraph'),
    ('Einstein won Nobel for photoelectric effect',
                                            'Einstein won Nobel for relativity'),
    ('Darwin wrote Origin of Species',      'Darwin wrote Descent of Man'),
    ('Washington was first president',      'Jefferson was first president'),
    ('Columbus sailed in 1492',             'Columbus sailed in 1493'),
]

PARAPHRASE_REPHRASED = [
    ('The telephone was invented by Bell',  'The telegraph was invented by Bell'),
    ('Einstein received the Nobel Prize for photoelectric effect',
                                            'Einstein received the Nobel Prize for relativity'),
    ('The Origin of Species was written by Darwin',
                                            'Descent of Man was written by Darwin'),
    ('The first president was Washington',  'The first president was Jefferson'),
    ('In 1492 Columbus sailed',             'In 1493 Columbus sailed'),
]

MATH_BASE = [
    ('10 + 3 = 13',    '10 + 3 = 14'),
    ('700 - 12 = 688', '700 - 12 = 689'),
    ('-1 * 45 = -45',  '-1 * 45 = -54'),
    ('22 + 753 = 775', '22 + 753 = 757'),
]

MATH_REPHRASED = [
    ('3 + 10 = 13',    '3 + 10 = 14'),
    ('688 = 700 - 12', '689 = 700 - 12'),
    ('45 * -1 = -45',  '45 * -1 = -54'),
    ('753 + 22 = 775', '753 + 22 = 757'),
]


def _signals_for_statement(text, mdl, tok, layer=21):
    """Return (gini, hidden_variance) for a single statement at one layer."""
    out = get_all_outputs(text, mdl, tok)
    att = out['attentions'][layer]
    hs  = out['hidden_states'][layer + 1]
    return compute_layer_gini(att), compute_hidden_variance(hs)


def run_paraphrase_experiment(base_pairs, rephrased_pairs, mdl, tok,
                               label='paraphrase', layer=21):
    """Measure how much structural signals change across paraphrases.

    Smaller change = signals more robust to surface form.
    Returns a DataFrame with robustness metrics per pair.
    """
    rows = []
    for i, ((t_orig, f_orig), (t_para, f_para)) in enumerate(
            zip(base_pairs, rephrased_pairs)):
        print(f'  Pair {i+1}: {t_orig[:40]}')
        t_og, t_ov = _signals_for_statement(t_orig, mdl, tok, layer)
        f_og, f_ov = _signals_for_statement(f_orig, mdl, tok, layer)
        t_pg, t_pv = _signals_for_statement(t_para, mdl, tok, layer)
        f_pg, f_pv = _signals_for_statement(f_para, mdl, tok, layer)

        rows.append({
            'pair_id'                      : i,
            'true_original'                : t_orig,
            'false_original'               : f_orig,
            'true_paraphrase'              : t_para,
            'false_paraphrase'             : f_para,
            # Raw signals
            'true_orig_gini'               : t_og,
            'false_orig_gini'              : f_og,
            'true_para_gini'               : t_pg,
            'false_para_gini'              : f_pg,
            'true_orig_variance'           : t_ov,
            'false_orig_variance'          : f_ov,
            'true_para_variance'           : t_pv,
            'false_para_variance'          : f_pv,
            # Robustness: how much did the signal shift across paraphrases?
            'true_gini_robustness'         : abs(t_og - t_pg),
            'false_gini_robustness'        : abs(f_og - f_pg),
            'true_variance_robustness'     : abs(t_ov - t_pv),
            'false_variance_robustness'    : abs(f_ov - f_pv),
            'gini_more_robust'             : 'true' if abs(t_og-t_pg) < abs(f_og-f_pg) else 'false',
            'variance_more_robust'         : 'true' if abs(t_ov-t_pv) < abs(f_ov-f_pv) else 'false',
        })

    df = pd.DataFrame(rows)

    print(f'\n{"="*60}')
    print(f'PARAPHRASE ROBUSTNESS: {label}')
    print(f'{"="*60}')
    true_gini_robust = (df['gini_more_robust'] == 'true').sum()
    true_var_robust  = (df['variance_more_robust'] == 'true').sum()
    n = len(df)
    print(f'True statements more robust: gini={true_gini_robust}/{n}  '
          f'variance={true_var_robust}/{n}')
    print(f'(NOTE: n={n} is too small for reliable inference — '
          f'treat as exploratory only)')

    from scipy.stats import binomtest, ttest_rel
    for label2, k in [('gini', true_gini_robust), ('variance', true_var_robust)]:
        p = binomtest(k, n, 0.5).pvalue
        print(f'  Binomial test ({label2} true more robust): p={p:.4f} '
              f'{"*" if p < 0.05 else "ns"}')
    return df


print('Running paraphrase test — factual pairs:')
df_paraphrase = run_paraphrase_experiment(
    PARAPHRASE_BASE, PARAPHRASE_REPHRASED, model, tokenizer,
    label='factual paraphrases')
df_paraphrase.to_csv('results_paraphrase.csv', index=False)

# Math equivalency pairs
# BLOG NOTE: These results are not mentioned in the blog. Run them and
# decide whether to add a brief note or remove the data entirely.
print('\nRunning paraphrase test — math equivalency pairs:')
df_math_para = run_paraphrase_experiment(
    MATH_BASE, MATH_REPHRASED, model, tokenizer,
    label='math equivalencies')
df_math_para.to_csv('results_math_paraphrase.csv', index=False)

In [ ]:
# ── SECTION 15: QWEN2.5-0.5B TESTING (Blog §VI) ─────────────────────────────
#
# PURPOSE: Test whether entropy patterns from GPT-2 Medium generalise to a
# more modern architecture. Blog conclusion: they do not — Qwen showed
# consistently stable entropy regardless of content accuracy.
#
# ── METHODOLOGICAL DIFFERENCES VS GPT-2 (IMPORTANT for blog) ─────────────────
#
# 1. ENTROPY IS NORMALISED HERE, NOT IN GPT-2 SECTIONS
#    Qwen entropy = raw_entropy / log(seq_len), giving a 0-1 scale.
#    GPT-2 entropy is in raw nats (typically 0.7-1.0+).
#    The numbers cannot be compared directly.
#    BLOG NOTE: The blog should clarify this when comparing entropy ranges
#    across models. "Qwen showed entropy 0.68-0.82" vs "GPT-2 Medium showed
#    entropy 0.75-1.0+" are on different scales.
#
# 2. ONLY THE FINAL TOKEN'S ATTENTION ROW IS USED
#    Qwen: entropy computed from attention_weights[:, -1, :] — the last
#    token's attention distribution only.
#    GPT-2: entropy averaged across all sequence positions.
#    These measure different things. GPT-2 asks "how dispersed is attention
#    across the whole sequence?" Qwen asks "how dispersed is the final
#    token's attention?" This should be unified in future work.
#    BLOG NOTE: Acknowledge that the measurement approach differs and this
#    limits the validity of cross-model comparison.
#
# 3. INSTRUCT MODEL, NOT BASE MODEL
#    This uses Qwen2.5-0.5B-Instruct (fine-tuned for instruction following),
#    not the base language model. Instruct fine-tuning changes the internal
#    distribution of attention patterns. For a fair comparison with GPT-2
#    (a base model) the Qwen base model should be used.
#    BLOG NOTE: Flag this as a limitation. Results may reflect instruction
#    tuning as much as architectural differences.
#
# 4. TEST PROMPTS ARE SIMPLER THAN GPT-2 EXPERIMENTS
#    Only five easy baseline prompts were run. The blog mentions biographical
#    prompts ("Tell me about Alexander Graham Bell") but those do not appear
#    in this notebook. Either those results came from a separate run not
#    captured here, or were extrapolated from the baseline results.
#    BLOG NOTE: The biographical prompt results cited in §VI need to be
#    verified or removed. If they cannot be reproduced, say so.
#
# 5. GENERATION IS TOKEN-BY-TOKEN (autoregressive loop)
#    Unlike GPT-2 tests which fed complete sequences, this generates one
#    token at a time and inspects attention after each step. This is closer
#    to real inference but makes direct comparison with GPT-2 forward-pass
#    tests harder to interpret.
#
# ── WHAT THIS TELLS US ───────────────────────────────────────────────────────
# Even with the methodological differences, the qualitative conclusion is
# probably sound: Qwen's normalised entropy stayed stable (0.68-0.82) across
# both correct and incorrect outputs, providing no discriminative signal.
# This supports the blog's conclusion that GPT-2 Medium's entropy patterns
# did not transfer to a different architecture — though "different measurement
# method" cannot be fully ruled out as a contributing factor.
# ─────────────────────────────────────────────────────────────────────────────

from transformers import AutoModelForCausalLM, AutoTokenizer

def load_qwen(model_name='Qwen/Qwen2.5-0.5B-Instruct'):
    """Load Qwen model with eager attention for weight extraction.

    Uses float32 for CPU compatibility; on GPU this can be changed to
    torch.float16 for memory efficiency.
    Note: trust_remote_code=True is required for Qwen tokenizer.
    """
    print(f'Loading {model_name}...')
    mdl = AutoModelForCausalLM.from_pretrained(
        model_name,
        attn_implementation='eager',
        trust_remote_code=True,
        torch_dtype=torch.float32
    ).to(DEVICE)
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mdl.config.output_attentions = True
    mdl.eval()
    n_layers = mdl.config.num_hidden_layers
    n_heads  = mdl.config.num_attention_heads
    print(f'  Layers: {n_layers}, Heads: {n_heads}')
    return mdl, tok


def qwen_entropy_normalised(attention_layer, seq_len):
    """Normalised Shannon entropy of the final token's attention row.

    attention_layer shape: (heads, seq_len, seq_len)
    Returns value in [0, 1] where 1 = maximally dispersed.

    NOTE: This differs from compute_layer_entropy() used in GPT-2 sections,
    which averages entropy across all positions and returns raw nats.
    Results are NOT directly comparable numerically.
    """
    final_row = attention_layer[:, -1, :] + 1e-10   # (heads, seq_len)
    h = -(final_row * torch.log(final_row)).sum(dim=1)  # (heads,)
    mean_h = h.mean().item()
    max_h  = np.log(seq_len)
    return mean_h / max_h if max_h > 0 else 0.0


def run_qwen_generation(prompt, mdl, tok, max_tokens=50, layers=None):
    """Generate token-by-token and record normalised entropy per step.

    layers: list of layer indices to monitor.
            Defaults to the last 3 layers of the model.
    Returns a DataFrame with one row per generated token.
    """
    if layers is None:
        n = mdl.config.num_hidden_layers
        layers = [n - 3, n - 2, n - 1]

    input_ids = tok.encode(prompt, return_tensors='pt').to(DEVICE)
    rows = []

    for step in range(max_tokens):
        with torch.no_grad():
            out = mdl(input_ids, output_attentions=True, return_dict=True)

        next_id = torch.argmax(out.logits[0, -1, :]).unsqueeze(0).unsqueeze(0)
        token_text = tok.decode(next_id[0])
        seq_len = input_ids.shape[1]

        row = {'token_num': step + 1, 'token_text': token_text}
        for layer_idx in layers:
            att = out.attentions[layer_idx][0]   # (heads, seq, seq)
            row[f'entropy_L{layer_idx}'] = qwen_entropy_normalised(att, seq_len)
        rows.append(row)

        input_ids = torch.cat([input_ids, next_id], dim=1)
        if next_id.item() == tok.eos_token_id:
            break

    return pd.DataFrame(rows)


# ── TEST PROMPTS ─────────────────────────────────────────────────────────────
# These are the baseline prompts from the original notebook.
# BLOG NOTE: If biographical prompts were also run, add them here and
# re-run to get reproducible results. Otherwise remove those claims from §VI.

QWEN_TEST_PROMPTS = [
    ('Water freezes at',            'factual_easy'),
    ('The sky is blue because',     'factual_easy'),
    ('Dogs are animals that',       'factual_easy'),
    ('2 + 2 equals',                'arithmetic_easy'),
    ('The capital of France is',    'factual_easy'),
]

# ── RUN ──────────────────────────────────────────────────────────────────────
model_qwen, tokenizer_qwen = load_qwen('Qwen/Qwen2.5-0.5B-Instruct')

qwen_results = []
for prompt, category in QWEN_TEST_PROMPTS:
    print(f'\n{\'=\'*60}')
    print(f'Prompt: {prompt}')
    df = run_qwen_generation(prompt, model_qwen, tokenizer_qwen, max_tokens=50)
    df['prompt'] = prompt
    df['category'] = category
    qwen_results.append(df)

    layer_cols = [c for c in df.columns if c.startswith('entropy_L')]
    mid = layer_cols[len(layer_cols) // 2]
    print(f'  Mean entropy ({mid}): {df[mid].mean():.3f}')
    print(f'  Max entropy         : {df[mid].max():.3f}')
    print(f'  Generated: {prompt + \'\'.join(df[\'token_text\'].tolist())[:80]}')

df_qwen = pd.concat(qwen_results, ignore_index=True)
df_qwen.to_csv('results_qwen.csv', index=False)

# ── SUMMARY ──────────────────────────────────────────────────────────────────
print('\nQwen entropy summary (normalised 0-1 scale):')
layer_cols = [c for c in df_qwen.columns if c.startswith('entropy_L')]
for col in layer_cols:
    print(f'  {col}: mean={df_qwen[col].mean():.3f}  '
          f'min={df_qwen[col].min():.3f}  max={df_qwen[col].max():.3f}')
print('\nNote: compare to GPT-2 raw entropy values with caution (different scales).')
